# Data Pipeline: CSV to MySQL with Polars

This notebook demonstrates a complete data engineering pipeline for loading multiple CSV files into a MySQL database using **Polars** and **SQLAlchemy**. The approach leverages Polars' lazy API for efficient memory usage and respects referential integrity constraints during the load process.

## 1. Library Imports

The following libraries are imported for data manipulation, database connectivity, and file system operations.

In [21]:
import polars as pl
import os
from pathlib import Path
from sqlalchemy import create_engine, text

## 2. Environment Configuration

Define the data source directory and database connection parameters. The connection uses the PyMySQL driver for MySQL compatibility.

In [22]:
# Data source directory
data_path = Path.cwd() / "archive"

# MySQL connection configuration (running in container)
DB_USER = "root"
DB_PASSWORD = "rootpass"
DB_HOST = "localhost"
DB_PORT = "3306"
DB_NAME = "Ecommerce"

# SQLAlchemy connection URL
database_url = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# Connection engine
engine = create_engine(database_url)

## 3. Database Schema Inspection

Query the MySQL metadata tables to inspect the current structure of the target database.

In [23]:
schema_query = """SELECT 
    TABLE_NAME, 
    COLUMN_NAME, 
    DATA_TYPE, 
    IS_NULLABLE, 
    COLUMN_KEY, 
    COLUMN_DEFAULT
FROM 
    INFORMATION_SCHEMA.COLUMNS
WHERE 
    TABLE_SCHEMA = 'Ecommerce'
ORDER BY 
    TABLE_NAME, ORDINAL_POSITION;"""

with engine.connect() as conn:
    schema_df = pl.read_database(query=schema_query, connection=conn)
    print(schema_df)

shape: (52, 6)
┌────────────┬──────────────────────────┬───────────┬─────────────┬────────────┬────────────────┐
│ TABLE_NAME ┆ COLUMN_NAME              ┆ DATA_TYPE ┆ IS_NULLABLE ┆ COLUMN_KEY ┆ COLUMN_DEFAULT │
│ ---        ┆ ---                      ┆ ---       ┆ ---         ┆ ---        ┆ ---            │
│ str        ┆ str                      ┆ str       ┆ str         ┆ str        ┆ null           │
╞════════════╪══════════════════════════╪═══════════╪═════════════╪════════════╪════════════════╡
│ customers  ┆ customer_id              ┆ char      ┆ NO          ┆ PRI        ┆ null           │
│ customers  ┆ customer_unique_id       ┆ char      ┆ NO          ┆            ┆ null           │
│ customers  ┆ customer_zip_code_prefix ┆ varchar   ┆ NO          ┆            ┆ null           │
│ customers  ┆ customer_city            ┆ varchar   ┆ NO          ┆            ┆ null           │
│ customers  ┆ customer_state           ┆ char      ┆ NO          ┆            ┆ null           │
│ …  

## 4. Lazy CSV Loading

Files are loaded lazily using `pl.scan_csv()`, which creates an execution plan without reading data into memory. This approach is memory-efficient for large datasets and allows for optimization before execution.

In [24]:
# Dictionary to store lazy frames
lazy_dfs = {}

for file_path in data_path.iterdir():
    if file_path.is_file() and file_path.suffix == ".csv":
        # Clean filename to match table name
        table_name = file_path.stem.replace("olist_", "").replace("product_category_name_", "").replace("_dataset", "")
        
        # Store lazy frame
        lazy_dfs[table_name] = pl.scan_csv(file_path)
        print(f"Loaded (lazy): {table_name}")

Loaded (lazy): products
Loaded (lazy): sellers
Loaded (lazy): customers
Loaded (lazy): orders
Loaded (lazy): order_items
Loaded (lazy): order_reviews
Loaded (lazy): translation
Loaded (lazy): order_payments
Loaded (lazy): geolocation


## 5. Load Order Definition

Define the correct order for loading tables to respect foreign key constraints. Dimension tables must be loaded before fact tables.

In [25]:
load_order = [
    # Dimension tables (independent)
    "customers",
    "products",
    "sellers",
    "geolocation",
    # Fact tables (dependent)
    "orders",
    "order_items",
    "order_payments",
    "order_reviews",
]

## 6. Data Load Pipeline

Execute the ingestion pipeline with idempotency checks. Tables already containing data are skipped to prevent duplication.

In [26]:
for table_name in load_order:
    print(f"Processing table: {table_name}...")
    
    # Check if table already contains data
    with engine.connect() as conn:
        row_count = conn.execute(text(f"SELECT COUNT(*) FROM {table_name}")).scalar()
    
    if row_count > 0:
        print(f"⏭️ Table '{table_name}' has {row_count:,} rows. Skipping...\n")
        continue
    
    if table_name in lazy_dfs:
        print(f"⬆️ Loading data into '{table_name}'...")
        
        # Materialize and load
        df = lazy_dfs[table_name].collect()
        df.write_database(
            table_name=table_name,
            connection=engine,
            if_table_exists="append"
        )
        
        print(f"✅ Table '{table_name}' loaded successfully.\n")
    else:
        print(f"⚠️ Table '{table_name}' not found in data dictionary.\n")

Processing table: customers...
⏭️ Table 'customers' has 99,441 rows. Skipping...

Processing table: products...
⏭️ Table 'products' has 32,951 rows. Skipping...

Processing table: sellers...
⏭️ Table 'sellers' has 3,095 rows. Skipping...

Processing table: geolocation...
⏭️ Table 'geolocation' has 1,000,163 rows. Skipping...

Processing table: orders...
⏭️ Table 'orders' has 99,441 rows. Skipping...

Processing table: order_items...
⏭️ Table 'order_items' has 112,650 rows. Skipping...

Processing table: order_payments...
⏭️ Table 'order_payments' has 103,886 rows. Skipping...

Processing table: order_reviews...
⏭️ Table 'order_reviews' has 99,224 rows. Skipping...



## 7. Validation Queries

Execute analytical queries to validate data integrity and demonstrate the database's analytical capabilities.

### Query 1: Order Details

Retrieve detailed order information including customer, product, and seller data for delivered orders.

In [27]:
query1 = """
SELECT 
    o.order_id,
    c.customer_id,
    c.customer_city,
    c.customer_state,
    o.order_status,
    i.order_item_id,
    p.product_category_name,
    i.price,
    i.freight_value,
    s.seller_id,
    s.seller_city
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id
INNER JOIN order_items i ON o.order_id = i.order_id
INNER JOIN products p ON i.product_id = p.product_id
INNER JOIN sellers s ON i.seller_id = s.seller_id
WHERE o.order_status = 'delivered'
LIMIT 100;
"""

with engine.connect() as conn:
    order_details = pl.read_database(query=query1, connection=conn)
    print(order_details)

shape: (100, 11)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ order_id  ┆ customer_ ┆ customer_ ┆ customer_ ┆ … ┆ price     ┆ freight_v ┆ seller_id ┆ seller_c │
│ ---       ┆ id        ┆ city      ┆ state     ┆   ┆ ---       ┆ alue      ┆ ---       ┆ ity      │
│ str       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ decimal[3 ┆ ---       ┆ str       ┆ ---      │
│           ┆ str       ┆ str       ┆ str       ┆   ┆ 8,2]      ┆ decimal[3 ┆           ┆ str      │
│           ┆           ┆           ┆           ┆   ┆           ┆ 8,2]      ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 00010242f ┆ 3ce436f18 ┆ campos    ┆ RJ        ┆ … ┆ 58.90     ┆ 13.29     ┆ 48436dade ┆ volta    │
│ e8c5a6d1b ┆ 3e68e0787 ┆ dos goyta ┆           ┆   ┆           ┆           ┆ 18ac8b2bc ┆ redonda  │
│ a2dd792cb ┆ 7b285a838 ┆ cazes     ┆           ┆   ┆           ┆         

### Query 2: Payment Summary

Aggregate payment and review data for orders with total paid amount exceeding 1000, sorted by highest value first.

In [28]:
query2 = """ 
SELECT 
    o.order_id,
    o.order_status,
    COUNT(DISTINCT p.payment_sequential) AS payment_count,
    SUM(p.payment_value) AS total_paid_amount,
    AVG(r.review_score) AS avg_review_score,
    MAX(r.review_creation_date) AS last_review_date
FROM orders o
LEFT JOIN order_payments p ON o.order_id = p.order_id
LEFT JOIN order_reviews r ON o.order_id = r.order_id
GROUP BY o.order_id, o.order_status
HAVING total_paid_amount > 1000 
ORDER BY total_paid_amount DESC
LIMIT 50;
"""

with engine.connect() as conn:
    payment_summary = pl.read_database(query=query2, connection=conn)
    print(payment_summary)

shape: (50, 6)
┌────────────────┬──────────────┬───────────────┬────────────────┬────────────────┬────────────────┐
│ order_id       ┆ order_status ┆ payment_count ┆ total_paid_amo ┆ avg_review_sco ┆ last_review_da │
│ ---            ┆ ---          ┆ ---           ┆ unt            ┆ re             ┆ te             │
│ str            ┆ str          ┆ i64           ┆ ---            ┆ ---            ┆ ---            │
│                ┆              ┆               ┆ decimal[38,2]  ┆ decimal[38,4]  ┆ datetime[μs]   │
╞════════════════╪══════════════╪═══════════════╪════════════════╪════════════════╪════════════════╡
│ 03caa2c082116e ┆ delivered    ┆ 1             ┆ 13664.08       ┆ 1.0000         ┆ 2017-10-18     │
│ 1d31e67e9ae370 ┆              ┆               ┆                ┆                ┆ 00:00:00       │
│ 04…            ┆              ┆               ┆                ┆                ┆                │
│ 736e1922ae60d0 ┆ delivered    ┆ 1             ┆ 7274.88        ┆ 1.0000   

### Query 3: Missing Geolocation Data

Identify customers whose zip code prefixes do not have corresponding entries in the geolocation table.

In [29]:
query3 = """ 
SELECT 
    c.customer_zip_code_prefix,
    c.customer_city,
    c.customer_state,
    COUNT(c.customer_id) AS customers_without_geolocation
FROM customers c
LEFT JOIN geolocation g ON c.customer_zip_code_prefix = g.geolocation_zip_code_prefix
WHERE g.geolocation_zip_code_prefix IS NULL
GROUP BY c.customer_zip_code_prefix, c.customer_city, c.customer_state
ORDER BY customers_without_geolocation DESC;
"""

with engine.connect() as conn:
    missing_geo = pl.read_database(query=query3, connection=conn)
    print(missing_geo)

shape: (157, 4)
┌──────────────────────────┬──────────────────────────┬────────────────┬───────────────────────────┐
│ customer_zip_code_prefix ┆ customer_city            ┆ customer_state ┆ customers_without_geoloca │
│ ---                      ┆ ---                      ┆ ---            ┆ tion                      │
│ str                      ┆ str                      ┆ str            ┆ ---                       │
│                          ┆                          ┆                ┆ i64                       │
╞══════════════════════════╪══════════════════════════╪════════════════╪═══════════════════════════╡
│ 70686                    ┆ brasilia                 ┆ DF             ┆ 15                        │
│ 72005                    ┆ brasilia                 ┆ DF             ┆ 13                        │
│ 71919                    ┆ brasilia                 ┆ DF             ┆ 10                        │
│ 73255                    ┆ brasilia                 ┆ DF             ┆ 7 